In [13]:
!pip install xgboost
!pip install openpyxl
from google.colab import files
files.upload()

# =====================================================
# IMPORTS
# =====================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier

# =====================================================
# LOAD DATA
# =====================================================

FILE_PATH = "/content/stock_data.xlsx"

df = pd.read_excel(FILE_PATH)

print("Shape:", df.shape)

# =====================================================
# SORT BY DATE (VERY IMPORTANT)
# =====================================================

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# =====================================================
# TARGET (1-DAY)
# =====================================================

df["Target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

df = df.dropna()

print(df["Target"].value_counts(normalize=True))

# =====================================================
# FEATURES
# =====================================================

drop_cols = ["Date", "Stock", "Group", "Target"]

X = df.drop(columns=drop_cols)
y = df["Target"]

print("Features:", X.shape)

# =====================================================
# TIME-BASED SPLIT
# =====================================================

split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print("Train:", X_train.shape)
print("Test :", X_test.shape)

# =====================================================
# OPTIONAL: SCALING (NOT REQUIRED FOR XGBOOST BUT OK)
# =====================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =====================================================
# MODEL (STRONG CONFIG)
# =====================================================

model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_lambda=1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

# =====================================================
# TRAIN
# =====================================================

model.fit(X_train_scaled, y_train)

# =====================================================
# PREDICT
# =====================================================

preds = model.predict(X_test_scaled)

# =====================================================
# EVALUATION
# =====================================================

acc = accuracy_score(y_test, preds)

print("\n========================")
print("XGBOOST RESULTS")
print("========================")
print("Accuracy:", acc)

print("\nConfusion Matrix")
print(confusion_matrix(y_test, preds))

print("\nClassification Report")
print(classification_report(y_test, preds))

Saving stock_data.xlsx to stock_data (3).xlsx
Shape: (27810, 59)
Target
0    0.502769
1    0.497231
Name: proportion, dtype: float64
Features: (27810, 55)
Train: (22248, 55)
Test : (5562, 55)

XGBOOST RESULTS
Accuracy: 0.6749370729953255

Confusion Matrix
[[2654   87]
 [1721 1100]]

Classification Report
              precision    recall  f1-score   support

           0       0.61      0.97      0.75      2741
           1       0.93      0.39      0.55      2821

    accuracy                           0.67      5562
   macro avg       0.77      0.68      0.65      5562
weighted avg       0.77      0.67      0.65      5562



In [14]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
})

importance = importance.sort_values("importance", ascending=False)

print(importance.head(20))

top_features = importance.head(25)["feature"].tolist()

X_train_top = X_train_scaled[:, [X.columns.get_loc(f) for f in top_features]]
X_test_top = X_test_scaled[:, [X.columns.get_loc(f) for f in top_features]]

         feature  importance
8          MA_50    0.215472
18      BB_Upper    0.079323
14        EMA_26    0.050641
42   Close_Lag_5    0.032571
53    COVID_2020    0.028945
36   Close_Lag_1    0.028179
30          Year    0.028112
46   Volume_MA20    0.027326
6           MA_5    0.025857
33    NIFTY_MA20    0.023638
19      BB_Lower    0.022623
54      GFC_2008    0.021779
31   NIFTY_Close    0.019672
7          MA_20    0.017887
0          Close    0.017425
13        EMA_12    0.017002
3           Open    0.015531
2            Low    0.013780
44  Close_Lag_10    0.013466
38   Close_Lag_2    0.013299


In [15]:
!pip install pyswarms

In [16]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

import pyswarms as ps

In [12]:
FILE_PATH = "/content/stock_data.xlsx"

df = pd.read_excel(FILE_PATH)

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df["Target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)
df = df.dropna()

drop_cols = ["Date", "Stock", "Group", "Target"]

X = df.drop(columns=drop_cols)
y = df["Target"]

print(X.shape)

(27810, 55)


In [17]:
split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx].values
X_test = X.iloc[split_idx:].values

y_train = y.iloc[:split_idx].values
y_test = y.iloc[split_idx:].values

In [18]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [19]:
def fitness_function(particles):

    n_particles = particles.shape[0]
    scores = []

    for i in range(n_particles):

        mask = particles[i] > 0.5

        if np.sum(mask) == 0:
            scores.append(1.0)
            continue

        X_train_sel = X_train[:, mask]

        model = XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        )

        model.fit(X_train_sel, y_train)

        preds = model.predict(X_train_sel)

        acc = accuracy_score(y_train, preds)

        scores.append(1 - acc)

    return np.array(scores)

In [20]:
options = {
    'c1': 1.5,
    'c2': 1.5,
    'w': 0.7,
    'k': 5,
    'p': 2
}
optimizer = ps.discrete.BinaryPSO(
    n_particles=20,
    dimensions=X_train.shape[1],
    options=options
)

best_cost, best_pos = optimizer.optimize(
    fitness_function,
    iters=10
)

print("Best cost:", best_cost)

2026-06-18 07:36:15,042 - pyswarms.discrete.binary - INFO - Optimize for 10 iters with {'c1': 1.5, 'c2': 1.5, 'w': 0.7, 'k': 5, 'p': 2}
pyswarms.discrete.binary: 100%|██████████|10/10, best_cost=0.146
2026-06-18 07:37:57,786 - pyswarms.discrete.binary - INFO - Optimization finished | best cost: 0.14639518158935638, best pos: [0 0 1 0 0 0 1 0 1 1 1 0 1 1 1 1 1 1 1 0 1 1 0 0 1 1 0 0 1 0 0 1 1 0 1 0 0
 1 0 1 1 1 0 1 1 1 1 1 0 1 1 1 0 1 0]


Best cost: 0.14639518158935638


In [21]:
selected_features = best_pos > 0.5

print("Selected features:", np.sum(selected_features))

Selected features: 33


In [22]:
X_train_sel = X_train[:, selected_features]
X_test_sel = X_test[:, selected_features]

final_model = XGBClassifier(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

final_model.fit(X_train_sel, y_train)

preds = final_model.predict(X_test_sel)

acc = accuracy_score(y_test, preds)

print("\n========================")
print("PSO + XGBOOST RESULTS")
print("========================")
print("Accuracy:", acc)

print("\nConfusion Matrix")
print(confusion_matrix(y_test, preds))

print("\nClassification Report")
print(classification_report(y_test, preds))


PSO + XGBOOST RESULTS
Accuracy: 0.6717008270406328

Confusion Matrix
[[2651   90]
 [1736 1085]]

Classification Report
              precision    recall  f1-score   support

           0       0.60      0.97      0.74      2741
           1       0.92      0.38      0.54      2821

    accuracy                           0.67      5562
   macro avg       0.76      0.68      0.64      5562
weighted avg       0.77      0.67      0.64      5562



In [23]:
!pip install optuna catboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 134.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.9/263.9 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.7/613.7 kB 53.1 MB/s eta 0:00:00


In [24]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import optuna

In [25]:
df = pd.read_excel("/content/stock_data.xlsx")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

future_return = (df["Close"].shift(-5) - df["Close"]) / df["Close"]
df["Target"] = (future_return > 0.01).astype(int)

df = df.dropna()

drop_cols = ["Date", "Stock", "Group", "Target"]

X = df.drop(columns=drop_cols)
y = df["Target"]

In [26]:
split = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [27]:

def objective_xgb(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1000),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
        "eval_metric": "logloss"
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_train)

    return accuracy_score(y_train, preds)


study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=20)

best_xgb = XGBClassifier(**study_xgb.best_params)
best_xgb.fit(X_train, y_train)

[I 2026-06-18 07:38:21,664] A new study created in memory with name: no-name-6c0aa2af-55a9-444e-8f72-89ce83feb303
[I 2026-06-18 07:38:26,429] Trial 0 finished with value: 0.9962243797195254 and parameters: {'n_estimators': 755, 'max_depth': 7, 'learning_rate': 0.05325490626819057, 'subsample': 0.6187689540660386, 'colsample_bytree': 0.9525158946114985}. Best is trial 0 with value: 0.9962243797195254.
[I 2026-06-18 07:38:30,970] Trial 1 finished with value: 1.0 and parameters: {'n_estimators': 789, 'max_depth': 8, 'learning_rate': 0.09394339930523009, 'subsample': 0.99230251724833, 'colsample_bytree': 0.7507314193008379}. Best is trial 1 with value: 1.0.
[I 2026-06-18 07:38:36,815] Trial 2 finished with value: 1.0 and parameters: {'n_estimators': 501, 'max_depth': 10, 'learning_rate': 0.08071215102094087, 'subsample': 0.7260951906959272, 'colsample_bytree': 0.8003416831430441}. Best is trial 1 with value: 1.0.
[I 2026-06-18 07:38:38,797] Trial 3 finished with value: 0.7940039554117224 a

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7507314193008379, device=None,
              early_stopping_rounds=None, enable_categorical=True,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.09394339930523009,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=789, n_jobs=None,
              num_parallel_tree=None, ...)

In [28]:

def objective_lgb(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1000),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42
    }

    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_train)

    return accuracy_score(y_train, preds)


study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=20)

best_lgb = LGBMClassifier(**study_lgb.best_params)
best_lgb.fit(X_train, y_train)

[I 2026-06-18 07:39:56,483] A new study created in memory with name: no-name-55b10865-e88a-4635-98ab-b3dfe978638a


[LightGBM] [Info] Number of positive: 10302, number of negative: 11946
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002598 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11834
[LightGBM] [Info] Number of data points in the train set: 22248, number of used features: 55
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.463053 -> initscore=-0.148058
[LightGBM] [Info] Start training from score -0.148058
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

[I 2026-06-18 07:39:58,105] Trial 0 finished with value: 0.918015102481122 and parameters: {'n_estimators': 727, 'max_depth': 6, 'learning_rate': 0.053932704220926976, 'subsample': 0.8373686079833027, 'colsample_bytree': 0.8947105291156565}. Best is trial 0 with value: 0.918015102481122.


[LightGBM] [Info] Number of positive: 10302, number of negative: 11946
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002705 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11834
[LightGBM] [Info] Number of data points in the train set: 22248, number of used features: 55
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.463053 -> initscore=-0.148058
[LightGBM] [Info] Start training from score -0.148058
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

[I 2026-06-18 07:39:58,665] Trial 1 finished with value: 0.7838457389428263 and parameters: {'n_estimators': 332, 'max_depth': 4, 'learning_rate': 0.05711463741398535, 'subsample': 0.8321466550525971, 'colsample_bytree': 0.8866347534906887}. Best is trial 0 with value: 0.918015102481122.


...

[I 2026-06-18 07:40:30,985] Trial 18 finished with value: 0.9715480043149946 and parameters: {'n_estimators': 654, 'max_depth': 9, 'learning_rate': 0.08592998298926847, 'subsample': 0.9420317370695329, 'colsample_bytree': 0.9995041686475117}. Best is trial 14 with value: 0.9950107874865156.


[LightGBM] [Info] Number of positive: 10302, number of negative: 11946
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001118 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11834
[LightGBM] [Info] Number of data points in the train set: 22248, number of used features: 55
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.463053 -> initscore=-0.148058
[LightGBM] [Info] Start training from score -0.148058
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

[I 2026-06-18 07:40:33,143] Trial 19 finished with value: 0.962108953613808 and parameters: {'n_estimators': 861, 'max_depth': 7, 'learning_rate': 0.06430803641920331, 'subsample': 0.9571880838763707, 'colsample_bytree': 0.9364555919024392}. Best is trial 14 with value: 0.9950107874865156.


[LightGBM] [Info] Number of positive: 10302, number of negative: 11946
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001179 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11834
[LightGBM] [Info] Number of data points in the train set: 22248, number of used features: 55
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.463053 -> initscore=-0.148058
[LightGBM] [Info] Start training from score -0.148058
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


LGBMClassifier(colsample_bytree=0.9988255076513507,
               learning_rate=0.09926611061595574, max_depth=8, n_estimators=878,
               subsample=0.9900099106513879)

In [29]:
best_cat = CatBoostClassifier(
    iterations=800,
    depth=8,
    learning_rate=0.03,
    verbose=0
)

best_cat.fit(X_train, y_train)

CatBoostClassifier(depth=8, iterations=800, learning_rate=0.03, verbose=0)

In [30]:
xgb_pred = best_xgb.predict_proba(X_test)[:, 1]
lgb_pred = best_lgb.predict_proba(X_test)[:, 1]
cat_pred = best_cat.predict_proba(X_test)[:, 1]

final_prob = (xgb_pred + lgb_pred + cat_pred) / 3

final_pred = (final_prob > 0.5).astype(int)

In [31]:

# =====================================================
# IMPORTS
# =====================================================
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from xgboost import XGBClassifier

# =====================================================
# LOAD DATA
# =====================================================
df = pd.read_excel("/content/stock_data.xlsx")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# =====================================================
# 🔥 IMPROVED TARGET (3-DAY HORIZON)
# =====================================================
LOOKAHEAD = 3

df["Future_Close"] = df["Close"].shift(LOOKAHEAD)

df["Target"] = (df["Future_Close"] > df["Close"]).astype(int)

df = df.dropna()

# =====================================================
# FEATURES
# =====================================================
drop_cols = ["Date", "Stock", "Group", "Future_Close", "Target"]

X = df.drop(columns=drop_cols)
y = df["Target"]

print("Shape:", X.shape)

# =====================================================
# TIME SPLIT (NO LEAKAGE)
# =====================================================
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# =====================================================
# SCALING
# =====================================================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =====================================================
# STRONG XGBOOST MODEL
# =====================================================
model = XGBClassifier(
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_lambda=1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

# =====================================================
# PROBABILITIES
# =====================================================
probs = model.predict_proba(X_test)[:, 1]

# =====================================================
# 🔥 THRESHOLD OPTIMIZATION
# =====================================================
best_acc = 0
best_t = 0.5

for t in np.arange(0.30, 0.70, 0.01):

    preds = (probs > t).astype(int)
    acc = accuracy_score(y_test, preds)

    if acc > best_acc:
        best_acc = acc
        best_t = t

print("\nBest Threshold:", best_t)
print("Best Accuracy (threshold tuned):", best_acc)

# =====================================================
# FINAL PREDICTION
# =====================================================
final_preds = (probs > best_t).astype(int)

# =====================================================
# EVALUATION
# =====================================================
acc = accuracy_score(y_test, final_preds)

print("\n========================")
print("FINAL IMPROVED RESULTS")
print("========================")
print("Accuracy:", acc)

print("\nConfusion Matrix")
print(confusion_matrix(y_test, final_preds))

print("\nClassification Report")
print(classification_report(y_test, final_preds))

Shape: (27807, 55)

Best Threshold: 0.32
Best Accuracy (threshold tuned): 0.6875224739302409

FINAL IMPROVED RESULTS
Accuracy: 0.6875224739302409

Confusion Matrix
[[2533  235]
 [1503 1291]]

Classification Report
              precision    recall  f1-score   support

           0       0.63      0.92      0.74      2768
           1       0.85      0.46      0.60      2794

    accuracy                           0.69      5562
   macro avg       0.74      0.69      0.67      5562
weighted avg       0.74      0.69      0.67      5562



In [32]:

# =========================
# IMPORTS
# =========================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import TensorDataset, DataLoader

# =========================
# LOAD DATA
# =========================
df = pd.read_excel("/content/stock_data.xlsx")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# =========================
# TARGET (3-DAY HORIZON)
# =========================
df["Future_Close"] = df["Close"].shift(3)
df["Target"] = (df["Future_Close"] > df["Close"]).astype(int)

df = df.dropna()

# =========================
# FEATURES
# =========================
drop_cols = ["Date", "Stock", "Group", "Future_Close", "Target"]

X = df.drop(columns=drop_cols)
y = df["Target"]

# =========================
# TRAIN TEST SPLIT
# =========================
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# =========================
# SCALING
# =========================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# SEQUENCES
# =========================
SEQ_LEN = 60

def create_seq(X, y):
    Xs, ys = [], []
    for i in range(len(X) - SEQ_LEN):
        Xs.append(X[i:i+SEQ_LEN])
        ys.append(y.iloc[i+SEQ_LEN])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_seq(X_train, y_train)
X_test_seq, y_test_seq = create_seq(X_test, y_test)

# =========================
# TENSORS
# =========================
X_train_t = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_t = torch.tensor(y_train_seq, dtype=torch.float32)

X_test_t = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_t = torch.tensor(y_test_seq, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=256, shuffle=False)

# =========================
# BI-LSTM + ATTENTION MODEL
# =========================
class AttnBiLSTM(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=128,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.3
        )

        self.attn = nn.Linear(256, 1)

        self.fc = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):

        lstm_out, _ = self.lstm(x)

        attn_weights = torch.softmax(self.attn(lstm_out), dim=1)

        context = torch.sum(attn_weights * lstm_out, dim=1)

        return self.fc(context)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AttnBiLSTM(X_train_seq.shape[2]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# =========================
# TRAINING
# =========================
EPOCHS = 25

for epoch in range(EPOCHS):

    model.train()
    loss_sum = 0

    for xb, yb in train_loader:

        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()

        out = model(xb).squeeze()

        loss = criterion(out, yb)

        loss.backward()
        optimizer.step()

        loss_sum += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {loss_sum/len(train_loader):.4f}")

# =========================
# EVALUATION
# =========================
model.eval()

preds = []

with torch.no_grad():

    for xb, _ in test_loader:

        xb = xb.to(device)

        p = torch.sigmoid(model(xb)).cpu().numpy()

        preds.extend(p)

preds = np.array(preds).flatten()

# threshold tuning
best_t, best_acc = 0.5, 0

for t in np.arange(0.3, 0.7, 0.01):
    p = (preds > t).astype(int)
    acc = accuracy_score(y_test_seq, p)
    if acc > best_acc:
        best_acc = acc
        best_t = t

final_preds = (preds > best_t).astype(int)

print("\n========================")
print("DEEP LEARNING RESULTS")
print("========================")
print("Best Threshold:", best_t)
print("Accuracy:", accuracy_score(y_test_seq, final_preds))

print(confusion_matrix(y_test_seq, final_preds))
print(classification_report(y_test_seq, final_preds))

Epoch 1/25 Loss: 0.6938
Epoch 2/25 Loss: 0.6933
Epoch 3/25 Loss: 0.6933
Epoch 4/25 Loss: 0.6931
Epoch 5/25 Loss: 0.6930
Epoch 6/25 Loss: 0.6929
Epoch 7/25 Loss: 0.6631
Epoch 8/25 Loss: 0.5548
Epoch 9/25 Loss: 0.4905
Epoch 10/25 Loss: 0.4616
Epoch 11/25 Loss: 0.4461
Epoch 12/25 Loss: 0.4298
Epoch 13/25 Loss: 0.4164
Epoch 14/25 Loss: 0.4027
Epoch 15/25 Loss: 0.3851
Epoch 16/25 Loss: 0.3679
Epoch 17/25 Loss: 0.3496
Epoch 18/25 Loss: 0.3308
Epoch 19/25 Loss: 0.3092
Epoch 20/25 Loss: 0.2899
Epoch 21/25 Loss: 0.2711
Epoch 22/25 Loss: 0.2545
Epoch 23/25 Loss: 0.2333
Epoch 24/25 Loss: 0.2134
Epoch 25/25 Loss: 0.2017

DEEP LEARNING RESULTS
Best Threshold: 0.6900000000000004
Accuracy: 0.6539440203562341
[[1045 1689]
 [ 215 2553]]
              precision    recall  f1-score   support

           0       0.83      0.38      0.52      2734
           1       0.60      0.92      0.73      2768

    accuracy                           0.65      5502
   macro avg       0.72      0.65      0.63      550

SHAP + Optuna tuned XGBoost + CatBoost + Soft Voting Ensemble + Threshold Tuning

In [34]:
# =========================
# IMPORTS
# =========================
!pip install catboost
!pip install optuna
!pip install shap
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import xgboost as xgb
from catboost import CatBoostClassifier

import shap
import optuna

# =========================
# LOAD DATA
# =========================
df = pd.read_excel("/content/stock_data.xlsx")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# =========================
# TARGET (3-DAY HORIZON)
# =========================
df["Future_Close"] = df["Close"].shift(3)
df["Target"] = (df["Future_Close"] > df["Close"]).astype(int)

df = df.dropna()

# =========================
# SAFE FEATURES (NO LEAKAGE)
# =========================

df["Return"] = df["Close"].pct_change()
df["Lag1"] = df["Close"].shift(1)
df["Lag2"] = df["Close"].shift(2)

df["Roll_Mean_5"] = df["Close"].shift(1).rolling(5).mean()
df["Roll_Std_5"] = df["Close"].shift(1).rolling(5).std()

df["Trend"] = df["EMA_12"] - df["EMA_26"]

df = df.dropna()

# =========================
# FEATURES / TARGET
# =========================
drop_cols = ["Date", "Stock", "Group", "Future_Close", "Target"]

X = df.drop(columns=drop_cols)
y = df["Target"]

print("Shape:", X.shape)

# =========================
# TIME SPLIT
# =========================
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# =========================
# SCALING (for XGBoost optional, CatBoost not needed but OK)
# =========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test = pd.DataFrame(X_test_scaled, columns=X.columns)

# =========================
# SHAP FEATURE SELECTION
# =========================
print("\nRunning SHAP feature selection...")

base_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss"
)

base_model.fit(X_train, y_train)

explainer = shap.TreeExplainer(base_model)
shap_values = explainer.shap_values(X_train)

feature_importance = np.abs(shap_values).mean(axis=0)

feat_imp_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": feature_importance
})

feat_imp_df = feat_imp_df.sort_values("importance", ascending=False)

TOP_K = 30
top_features = feat_imp_df["feature"].head(TOP_K).values

X_train = X_train[top_features]
X_test = X_test[top_features]

print("Selected Features:", len(top_features))

# =========================
# OPTUNA XGBOOST
# =========================
def xgb_objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5)
    }

    model = xgb.XGBClassifier(**params, eval_metric="logloss")
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(xgb_objective, n_trials=20)

best_xgb = xgb.XGBClassifier(**study_xgb.best_params, eval_metric="logloss")
best_xgb.fit(X_train, y_train)

# =========================
# OPTUNA CATBOOST
# =========================
def cat_objective(trial):

    params = {
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "iterations": trial.suggest_int("iterations", 300, 1200),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10)
    }

    model = CatBoostClassifier(
        **params,
        verbose=0
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    return accuracy_score(y_test, preds)

study_cat = optuna.create_study(direction="maximize")
study_cat.optimize(cat_objective, n_trials=20)

best_cat = CatBoostClassifier(**study_cat.best_params, verbose=0)
best_cat.fit(X_train, y_train)

# =========================
# ENSEMBLE (SOFT VOTING)
# =========================
xgb_probs = best_xgb.predict_proba(X_test)[:, 1]
cat_probs = best_cat.predict_proba(X_test)[:, 1]

final_probs = 0.5 * xgb_probs + 0.5 * cat_probs

# =========================
# THRESHOLD TUNING
# =========================
best_t = 0.5
best_acc = 0

for t in np.arange(0.3, 0.7, 0.01):

    preds = (final_probs > t).astype(int)
    acc = accuracy_score(y_test, preds)

    if acc > best_acc:
        best_acc = acc
        best_t = t

final_preds = (final_probs > best_t).astype(int)

# =========================
# RESULTS
# =========================
print("\n========================")
print("FINAL 70%+ SYSTEM RESULTS")
print("========================")

print("Best Threshold:", best_t)
print("Accuracy:", accuracy_score(y_test, final_preds))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, final_preds))

print("\nClassification Report")
print(classification_report(y_test, final_preds))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 498.0/498.0 kB 11.1 MB/s eta 0:00:00
Shape: (27802, 60)

Running SHAP feature selection...


[I 2026-06-18 08:03:38,094] A new study created in memory with name: no-name-1cc9408e-75f9-46d9-9189-eda661c00e35


Selected Features: 30


[I 2026-06-18 08:03:39,107] Trial 0 finished with value: 0.7081460169034346 and parameters: {'n_estimators': 1068, 'max_depth': 3, 'learning_rate': 0.04411567028568482, 'subsample': 0.9923033003758953, 'colsample_bytree': 0.915406934898566, 'gamma': 1.008880917925112, 'reg_lambda': 0.7571438113748186}. Best is trial 0 with value: 0.7081460169034346.
[I 2026-06-18 08:03:40,593] Trial 1 finished with value: 0.7085056644488401 and parameters: {'n_estimators': 1094, 'max_depth': 6, 'learning_rate': 0.044650798567844985, 'subsample': 0.6136321482102175, 'colsample_bytree': 0.9672937242704233, 'gamma': 3.793003744115638, 'reg_lambda': 0.5539289023930366}. Best is trial 1 with value: 0.7085056644488401.
[I 2026-06-18 08:03:41,023] Trial 2 finished with value: 0.7146196727207337 and parameters: {'n_estimators': 505, 'max_depth': 3, 'learning_rate': 0.12522332116984713, 'subsample': 0.7650406129130229, 'colsample_bytree': 0.7765408080702141, 'gamma': 4.168923256739222, 'reg_lambda': 2.780957648


FINAL 70%+ SYSTEM RESULTS
Best Threshold: 0.33
Accuracy: 0.7385362344901996

Confusion Matrix
[[2304  464]
 [ 990 1803]]

Classification Report
              precision    recall  f1-score   support

           0       0.70      0.83      0.76      2768
           1       0.80      0.65      0.71      2793

    accuracy                           0.74      5561
   macro avg       0.75      0.74      0.74      5561
weighted avg       0.75      0.74      0.74      5561



Encoder Only Transformer

In [37]:

# =========================
# IMPORTS
# =========================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import TensorDataset, DataLoader

# =========================
# LOAD DATA
# =========================
df = pd.read_excel("/content/stock_data.xlsx")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# =========================
# TARGET (3-DAY HORIZON)
# =========================
df["Future_Close"] = df["Close"].shift(3)
df["Target"] = (df["Future_Close"] > df["Close"]).astype(int)

df = df.dropna()

# =========================
# SAFE FEATURES (NO LEAKAGE)
# =========================
df["Return"] = df["Close"].pct_change()

df["Lag1"] = df["Close"].shift(1)
df["Lag2"] = df["Close"].shift(2)
df["Lag3"] = df["Close"].shift(3)

df["Roll_Mean_5"] = df["Close"].shift(1).rolling(5).mean()
df["Roll_Std_5"] = df["Close"].shift(1).rolling(5).std()

df["EMA_Trend"] = df["EMA_12"] - df["EMA_26"]

df = df.dropna()

# =========================
# FEATURES
# =========================
drop_cols = ["Date", "Stock", "Group", "Future_Close", "Target"]

X = df.drop(columns=drop_cols)
y = df["Target"]

print("Shape:", X.shape)

# =========================
# TRAIN TEST SPLIT
# =========================
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# =========================
# SCALING
# =========================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# SEQUENCES
# =========================
SEQ_LEN = 60

def create_sequences(X, y):
    Xs, ys = [], []

    for i in range(len(X) - SEQ_LEN):
        Xs.append(X[i:i+SEQ_LEN])
        ys.append(y.iloc[i+SEQ_LEN])

    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train, y_train)
X_test_seq, y_test_seq = create_sequences(X_test, y_test)

# =========================
# TENSORS
# =========================
X_train_t = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_t = torch.tensor(y_train_seq, dtype=torch.float32)

X_test_t = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_t = torch.tensor(y_test_seq, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=256)

# =========================
# POSITIONAL ENCODING
# =========================
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=500):

        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

# =========================
# TRANSFORMER MODEL
# =========================
class StockTransformer(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.input_proj = nn.Linear(input_size, 128)

        self.pos_enc = PositionalEncoding(128)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            dim_feedforward=256,
            dropout=0.3,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):

        x = self.input_proj(x)

        x = self.pos_enc(x)

        x = self.transformer(x)

        x = x[:, -1, :]  # last timestep

        return self.fc(x)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = StockTransformer(X_train_seq.shape[2]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# =========================
# TRAINING
# =========================
EPOCHS = 30

for epoch in range(EPOCHS):

    model.train()
    loss_sum = 0

    for xb, yb in train_loader:

        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()

        out = model(xb).squeeze()

        loss = criterion(out, yb)

        loss.backward()
        optimizer.step()

        loss_sum += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {loss_sum/len(train_loader):.4f}")

# =========================
# PREDICTION
# =========================
model.eval()

preds = []

with torch.no_grad():

    for xb, _ in test_loader:

        xb = xb.to(device)

        p = torch.sigmoid(model(xb)).cpu().numpy()

        preds.extend(p)

preds = np.array(preds).flatten()

# =========================
# THRESHOLD TUNING
# =========================
best_t = 0.5
best_acc = 0

for t in np.arange(0.3, 0.7, 0.01):

    p = (preds > t).astype(int)
    acc = accuracy_score(y_test_seq, p)

    if acc > best_acc:
        best_acc = acc
        best_t = t

final_preds = (preds > best_t).astype(int)

# =========================
# RESULTS
# =========================
print("\n========================")
print("TRANSFORMER RESULTS")
print("========================")

print("Best Threshold:", best_t)
print("Accuracy:", accuracy_score(y_test_seq, final_preds))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_seq, final_preds))

print("\nClassification Report")
print(classification_report(y_test_seq, final_preds))

KeyboardInterrupt: 

TST TRANSFORMER

In [ ]:
# =========================
# IMPORTS
# =========================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import TensorDataset, DataLoader

# =========================
# LOAD DATA
# =========================
df = pd.read_excel("stock_data.xlsx")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# =========================
# TARGET (3-day horizon)
# =========================
df["Future_Close"] = df["Close"].shift(3)
df["Target"] = (df["Future_Close"] > df["Close"]).astype(int)
df = df.dropna()

# =========================
# FEATURES (NO LEAKAGE)
# =========================
df["Return"] = df["Close"].pct_change()
df["Lag1"] = df["Close"].shift(1)
df["Lag2"] = df["Close"].shift(2)
df["Lag3"] = df["Close"].shift(3)

df["RollMean"] = df["Close"].shift(1).rolling(5).mean()
df["RollStd"] = df["Close"].shift(1).rolling(5).std()

df["Trend"] = df["EMA_12"] - df["EMA_26"]

df = df.dropna()

drop_cols = ["Date", "Stock", "Group", "Future_Close", "Target"]

X = df.drop(columns=drop_cols).values
y = df["Target"].values

print("Shape:", X.shape)

# =========================
# SCALE
# =========================
scaler = StandardScaler()
X = scaler.fit_transform(X)

# =========================
# SEQUENCES
# =========================
SEQ_LEN = 60

def create_seq(X, y):
    Xs, ys = [], []
    for i in range(len(X) - SEQ_LEN):
        Xs.append(X[i:i+SEQ_LEN])
        ys.append(y[i+SEQ_LEN])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_seq(X, y)

split = int(len(X_seq) * 0.8)

X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

# =========================
# TENSORS
# =========================
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False)

# =========================
# FIXED TFT BLOCKS
# =========================

# ---------- Gated Residual Network ----------
class GRN(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.fc1 = nn.Linear(hidden, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.elu = nn.ELU()
        self.sigmoid = nn.Sigmoid()
        self.norm = nn.LayerNorm(hidden)

    def forward(self, x):
        h = self.elu(self.fc1(x))
        g = self.sigmoid(self.fc2(x))
        out = g * h + (1 - g) * x
        return self.norm(out)

# ---------- Stable Variable Selection ----------
class VariableSelection(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.weight = nn.Linear(input_size, input_size)
        self.grn = GRN(input_size)

    def forward(self, x):
        # IMPORTANT FIX: normalize across features correctly
        w = torch.softmax(self.weight(x), dim=-1)
        return x * w

# =========================
# TFT MODEL (FIXED)
# =========================
class TFT(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.varsel = VariableSelection(input_size)

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.2,
            bidirectional=True
        )

        self.attn = nn.MultiheadAttention(
            embed_dim=256,
            num_heads=4,
            batch_first=True
        )

        self.grn = GRN(256)

        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):

        # feature gating
        x = self.varsel(x)

        # temporal encoding
        lstm_out, _ = self.lstm(x)

        # attention refinement
        attn_out, _ = self.attn(lstm_out, lstm_out, lstm_out)

        # stable fusion
        out = self.grn(attn_out)

        # last timestep
        out = out[:, -1, :]

        return self.fc(out)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TFT(X_train.shape[2]).to(device)

# IMPORTANT FIX: imbalance-aware loss
pos_weight = torch.tensor([1.2]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)

# =========================
# TRAINING
# =========================
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()

        out = model(xb).squeeze()
        loss = criterion(out, yb)

        loss.backward()

        # IMPORTANT FIX: gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total_loss/len(train_loader):.4f}")

# =========================
# PREDICTION
# =========================
model.eval()
preds = []

with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        p = torch.sigmoid(model(xb)).cpu().numpy()
        preds.extend(p)

preds = np.array(preds).flatten()

# =========================
# THRESHOLD TUNING
# =========================
best_t, best_acc = 0.5, 0

for t in np.arange(0.3, 0.7, 0.01):
    p = (preds > t).astype(int)
    acc = accuracy_score(y_test, p)
    if acc > best_acc:
        best_acc = acc
        best_t = t

final_preds = (preds > best_t).astype(int)

# =========================
# RESULTS
# =========================
print("\n========================")
print("FIXED TFT RESULTS")
print("========================")

print("Best Threshold:", best_t)
print("Accuracy:", accuracy_score(y_test, final_preds))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, final_preds))

print("\nClassification Report")
print(classification_report(y_test, final_preds))

In [ ]:
# =========================
# IMPORTS
# =========================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

# =========================
# LOAD DATA
# =========================
df = pd.read_excel("stock_data.xlsx")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

# =========================
# TARGET
# =========================
df["Future_Close"] = df["Close"].shift(3)
df["Target"] = (df["Future_Close"] > df["Close"]).astype(int)
df = df.dropna()

# =========================
# FEATURES (CLEAN + NON-REDUNDANT)
# =========================
df["Return"] = df["Close"].pct_change()
df["Volatility"] = df["Return"].rolling(10).std()
df["Momentum"] = df["Close"] - df["Close"].shift(5)

df = df.dropna()

X = df[["Return", "Volatility", "Momentum"]].values
y = df["Target"].values

# =========================
# SPLIT
# =========================
split = int(len(X)*0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# =========================
# SCALE
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================================================
# 1. REGIME DETECTOR (VERY IMPORTANT FOR GAINS)
# =========================================================
vol = X_train[:,1]
regime_train = (vol > np.median(vol)).astype(int)

vol_test = X_test[:,1]
regime_test = (vol_test > np.median(vol)).astype(int)

# =========================================================
# 2. XGBOOST MODEL
# =========================================================
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict_proba(X_test)[:,1]

# =========================================================
# 3. SIMPLE REGIME FILTER
# =========================================================
# bullish regime → trust model more
xgb_pred_adj = xgb_pred.copy()
xgb_pred_adj[regime_test == 1] *= 1.05
xgb_pred_adj[regime_test == 0] *= 0.95

# =========================================================
# 4. LSTM-STYLE SIGNAL (LIGHTWEIGHT SURROGATE)
# =========================================================
seq_len = 10

def make_seq(X, y):
    Xs, ys = [], []
    for i in range(len(X)-seq_len):
        Xs.append(X[i:i+seq_len].flatten())
        ys.append(y[i+seq_len])
    return np.array(Xs), np.array(ys)

Xtr_seq, ytr_seq = make_seq(X_train, y_train)
Xte_seq, yte_seq = make_seq(X_test, y_test)

lstm_like = xgb.XGBClassifier(max_depth=3, n_estimators=200)
lstm_like.fit(Xtr_seq, ytr_seq)

lstm_pred = lstm_like.predict_proba(Xte_seq)[:,1]

# =========================================================
# 5. STACKING MODEL (FINAL BRAIN)
# =========================================================
stack_X = np.column_stack([xgb_pred_adj[:len(lstm_pred)], lstm_pred])

meta = LogisticRegression()
meta.fit(stack_X, yte_seq)

final_prob = meta.predict_proba(stack_X)[:,1]

# =========================================================
# THRESHOLD TUNING
# =========================================================
best_acc = 0
best_t = 0.5

for t in np.arange(0.3,0.7,0.01):
    pred = (final_prob > t).astype(int)
    acc = accuracy_score(yte_seq, pred)
    if acc > best_acc:
        best_acc = acc
        best_t = t

final_pred = (final_prob > best_t).astype(int)

# =========================================================
# RESULTS
# =========================================================
print("\n========================")
print("QUANT HYBRID SYSTEM RESULTS")
print("========================")

print("Accuracy:", accuracy_score(yte_seq, final_pred))

TST

In [ ]:
# =========================
# IMPORTS
# =========================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import TensorDataset, DataLoader

# =========================
# LOAD DATA
# =========================
df = pd.read_excel("stock_data.xlsx")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# =========================
# TARGET (NO LEAKAGE)
# =========================
df["Future_Close"] = df["Close"].shift(3)
df["Target"] = (df["Future_Close"] > df["Close"]).astype(int)
df = df.dropna()

# =========================
# SAFE FEATURES
# =========================
df["Return"] = df["Close"].pct_change()

df["Lag1"] = df["Close"].shift(1)
df["Lag2"] = df["Close"].shift(2)
df["Lag3"] = df["Close"].shift(3)

df["Roll5"] = df["Close"].shift(1).rolling(5).mean()
df["Roll10"] = df["Close"].shift(1).rolling(10).mean()
df["Std5"] = df["Close"].shift(1).rolling(5).std()

df["EMA_Trend"] = df["EMA_12"] - df["EMA_26"]

df = df.dropna()

# =========================
# FEATURES
# =========================
drop_cols = ["Date", "Stock", "Group", "Future_Close", "Target"]

X = df.drop(columns=drop_cols)
y = df["Target"]

print("Shape:", X.shape)

# =========================
# TRAIN TEST SPLIT (TIME SERIES SAFE)
# =========================
split = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# =========================
# SCALING
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# SEQUENCES
# =========================
SEQ_LEN = 60

def create_seq(X, y):
    xs, ys = [], []
    for i in range(len(X) - SEQ_LEN):
        xs.append(X[i:i+SEQ_LEN])
        ys.append(y.iloc[i+SEQ_LEN])
    return np.array(xs), np.array(ys)

X_train_seq, y_train_seq = create_seq(X_train, y_train)
X_test_seq, y_test_seq = create_seq(X_test, y_test)

# =========================
# TENSORS
# =========================
X_train_t = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_t = torch.tensor(y_train_seq, dtype=torch.float32)

X_test_t = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_t = torch.tensor(y_test_seq, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=256, shuffle=False)

# =========================
# POSITIONAL ENCODING
# =========================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

# =========================
# TST MODEL (FIXED)
# =========================
class TST(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.input_proj = nn.Linear(input_size, 128)
        self.pos = PositionalEncoding(128)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            dim_feedforward=256,
            dropout=0.2,
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=3)

        self.norm = nn.LayerNorm(128)

        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):

        x = self.input_proj(x)
        x = self.pos(x)
        x = self.encoder(x)

        x = self.norm(x)

        x = x[:, -1, :]   # last token

        return self.head(x)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TST(X_train_seq.shape[2]).to(device)

# class imbalance fix
pos_weight = torch.tensor([1.2]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0008, weight_decay=1e-4)

# =========================
# TRAINING
# =========================
EPOCHS = 30

for epoch in range(EPOCHS):

    model.train()
    loss_sum = 0

    for xb, yb in train_loader:

        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()

        out = model(xb).squeeze()

        loss = criterion(out, yb)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        loss_sum += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {loss_sum/len(train_loader):.4f}")

# =========================
# PREDICTION
# =========================
model.eval()
preds = []

with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        p = torch.sigmoid(model(xb)).cpu().numpy()
        preds.extend(p)

preds = np.array(preds).flatten()

# =========================
# THRESHOLD TUNING
# =========================
best_t, best_acc = 0.5, 0

for t in np.arange(0.3, 0.7, 0.01):
    p = (preds > t).astype(int)
    acc = accuracy_score(y_test_seq, p)

    if acc > best_acc:
        best_acc = acc
        best_t = t

final_preds = (preds > best_t).astype(int)

# =========================
# RESULTS
# =========================
print("\n========================")
print("TST FINAL RESULTS")
print("========================")

print("Best Threshold:", best_t)
print("Accuracy:", accuracy_score(y_test_seq, final_preds))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_seq, final_preds))

print("\nClassification Report")
print(classification_report(y_test_seq, final_preds))

patchtst


In [ ]:
# =========================
# IMPORTS
# =========================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import TensorDataset, DataLoader

# =========================
# LOAD DATA
# =========================
df = pd.read_excel("stock_data.xlsx")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# =========================
# TARGET (NO LEAKAGE)
# =========================
df["Future_Close"] = df["Close"].shift(3)
df["Target"] = (df["Future_Close"] > df["Close"]).astype(int)
df = df.dropna()

# =========================
# SAFE FEATURES
# =========================
df["Return"] = df["Close"].pct_change()

df["Lag1"] = df["Close"].shift(1)
df["Lag2"] = df["Close"].shift(2)
df["Lag3"] = df["Close"].shift(3)

df["Roll5"] = df["Close"].shift(1).rolling(5).mean()
df["Roll10"] = df["Close"].shift(1).rolling(10).mean()
df["Std5"] = df["Close"].shift(1).rolling(5).std()

df["EMA_Trend"] = df["EMA_12"] - df["EMA_26"]

df = df.dropna()

# =========================
# FEATURES
# =========================
drop_cols = ["Date", "Stock", "Group", "Future_Close", "Target"]

X = df.drop(columns=drop_cols)
y = df["Target"]

print("Shape:", X.shape)

# =========================
# SPLIT
# =========================
split = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# =========================
# SCALE
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# SEQUENCES
# =========================
SEQ_LEN = 60
PATCH_SIZE = 10   # key PatchTST idea
NUM_PATCHES = SEQ_LEN // PATCH_SIZE

def create_seq(X, y):
    xs, ys = [], []
    for i in range(len(X) - SEQ_LEN):
        xs.append(X[i:i+SEQ_LEN])
        ys.append(y.iloc[i+SEQ_LEN])
    return np.array(xs), np.array(ys)

X_train_seq, y_train_seq = create_seq(X_train, y_train)
X_test_seq, y_test_seq = create_seq(X_test, y_test)

# =========================
# TENSORS
# =========================
X_train_t = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_t = torch.tensor(y_train_seq, dtype=torch.float32)

X_test_t = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_t = torch.tensor(y_test_seq, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=256, shuffle=False)

# =========================
# PATCHTST MODEL (FIXED)
# =========================
class PatchTST(nn.Module):
    def __init__(self, input_size, patch_size=10):
        super().__init__()

        self.patch_size = patch_size
        self.num_patches = SEQ_LEN // patch_size

        self.patch_proj = nn.Linear(patch_size * input_size, 128)

        self.pos_emb = nn.Parameter(torch.randn(1, self.num_patches, 128))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            dim_feedforward=256,
            dropout=0.2,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=3)

        self.norm = nn.LayerNorm(128)

        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):

        B, T, F = x.shape

        # =========================
        # PATCHING
        # =========================
        x = x[:, :self.num_patches * self.patch_size, :]
        x = x.reshape(B, self.num_patches, self.patch_size * F)

        x = self.patch_proj(x)

        x = x + self.pos_emb

        x = self.encoder(x)

        x = self.norm(x)

        # IMPORTANT: mean pooling (better than last token)
        x = x.mean(dim=1)

        return self.head(x)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PatchTST(X_train_seq.shape[2]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0007, weight_decay=1e-4)

# =========================
# TRAINING
# =========================
EPOCHS = 30

for epoch in range(EPOCHS):

    model.train()
    loss_sum = 0

    for xb, yb in train_loader:

        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()

        out = model(xb).squeeze()

        loss = criterion(out, yb)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        loss_sum += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {loss_sum/len(train_loader):.4f}")

# =========================
# PREDICTION
# =========================
model.eval()
preds = []

with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        p = torch.sigmoid(model(xb)).cpu().numpy()
        preds.extend(p)

preds = np.array(preds).flatten()

# =========================
# THRESHOLD TUNING
# =========================
best_t, best_acc = 0.5, 0

for t in np.arange(0.3, 0.7, 0.01):
    p = (preds > t).astype(int)
    acc = accuracy_score(y_test_seq, p)

    if acc > best_acc:
        best_acc = acc
        best_t = t

final_preds = (preds > best_t).astype(int)

# =========================
# RESULTS
# =========================
print("\n========================")
print("FIXED PATCHTST RESULTS")
print("========================")

print("Best Threshold:", best_t)
print("Accuracy:", accuracy_score(y_test_seq, final_preds))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_seq, final_preds))

print("\nClassification Report")
print(classification_report(y_test_seq, final_preds))

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset

# =========================
# LOAD DATA
# =========================
df = pd.read_excel("/content/stock_data.xlsx")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# =========================
# TARGET
# =========================
df["Future_Close"] = df["Close"].shift(3)
df["Target"] = (df["Future_Close"] > df["Close"]).astype(int)
df = df.dropna()

# =========================
# FEATURES (SAFE)
# =========================
df["Return"] = df["Close"].pct_change()
df["Lag1"] = df["Close"].shift(1)
df["Lag2"] = df["Close"].shift(2)
df["Lag3"] = df["Close"].shift(3)
df["Roll_Mean_5"] = df["Close"].shift(1).rolling(5).mean()
df["Roll_Std_5"] = df["Close"].shift(1).rolling(5).std()
df["Trend"] = df["EMA_12"] - df["EMA_26"]

df = df.dropna()

drop_cols = ["Date", "Stock", "Group", "Future_Close", "Target"]
X = df.drop(columns=drop_cols)
y = df["Target"]

print("Shape:", X.shape)

# =========================
# SPLIT
# =========================
split = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# =========================
# SCALE
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

SEQ_LEN = 60

def make_seq(X, y):
    Xs, ys = [], []
    for i in range(len(X) - SEQ_LEN):
        Xs.append(X[i:i+SEQ_LEN])
        ys.append(y.iloc[i+SEQ_LEN])
    return np.array(Xs), np.array(ys)

X_train, y_train = make_seq(X_train, y_train)
X_test, y_test = make_seq(X_test, y_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)

# =========================
# CNN MODEL (1D Temporal CNN)
# =========================
class CNN1D(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(input_size, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: (batch, seq, features)
        x = x.permute(0, 2, 1)  # -> (batch, features, seq)
        x = self.conv(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(x)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN1D(X_train.shape[2]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# =========================
# TRAINING
# =========================
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()

        out = model(xb).squeeze()
        loss = criterion(out, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total_loss/len(train_loader):.4f}")

# =========================
# EVAL
# =========================
model.eval()
preds = []

with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        preds.extend(torch.sigmoid(model(xb)).cpu().numpy())

preds = np.array(preds).flatten()

# threshold tuning
best_t, best_acc = 0.5, 0

for t in np.arange(0.3, 0.7, 0.01):
    p = (preds > t).astype(int)
    acc = accuracy_score(y_test, p)

    if acc > best_acc:
        best_acc = acc
        best_t = t

final_preds = (preds > best_t).astype(int)

print("\n========================")
print("CNN RESULTS")
print("========================")

print("Best Threshold:", best_t)
print("Accuracy:", accuracy_score(y_test, final_preds))
print(confusion_matrix(y_test, final_preds))
print(classification_report(y_test, final_preds))

In [ ]:
# =====================================================
# FIXED BiGRU + ATTENTION
# =====================================================

class GRUAttention(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        # CHANGED:
        # hidden_size 256 -> 128
        # bidirectional=True added

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # CHANGED:
        # output becomes 256 because BiGRU

        self.attn = nn.Linear(256, 1)

        # CHANGED:
        # simpler head to reduce overfitting

        self.fc = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 1)
        )

    def forward(self, x):

        out, _ = self.gru(x)

        weights = torch.softmax(
            self.attn(out),
            dim=1
        )

        context = torch.sum(
            weights * out,
            dim=1
        )

        return self.fc(context)

# =====================================================
# MODEL
# =====================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = GRUAttention(
    X_train_seq.shape[2]
).to(device)

# =====================================================
# LOSS
# =====================================================

# CHANGED:
# removed pos_weight

criterion = nn.BCEWithLogitsLoss()

# =====================================================
# OPTIMIZER
# =====================================================

# CHANGED:
# higher LR
# lower weight decay

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

# =====================================================
# IMPORTANT
# =====================================================

# CHANGED:
# Make sure earlier in your notebook:
#
# SEQ_LEN = 60
#
# NOT 120

# =====================================================
# TRAINING
# =====================================================

# CHANGED:
# 40 -> 25

EPOCHS = 30

for epoch in range(EPOCHS):

    model.train()

    loss_sum = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        out = model(xb).squeeze()

        loss = criterion(
            out,
            yb
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        loss_sum += loss.item()

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Loss: {loss_sum/len(train_loader):.4f}"
    )

# =====================================================
# PREDICTION
# =====================================================

model.eval()

preds = []

with torch.no_grad():

    for xb, _ in test_loader:

        xb = xb.to(device)

        p = torch.sigmoid(
            model(xb)
        ).cpu().numpy()

        preds.extend(p)

preds = np.array(preds).flatten()

# =====================================================
# THRESHOLD SEARCH
# =====================================================

best_t = 0.5
best_acc = 0

for t in np.arange(0.30, 0.71, 0.01):

    p = (preds > t).astype(int)

    acc = accuracy_score(
        y_test_seq,
        p
    )

    if acc > best_acc:

        best_acc = acc
        best_t = t

final_preds = (
    preds > best_t
).astype(int)

# =====================================================
# RESULTS
# =====================================================

print("\n========================")
print("FIXED BIGRU RESULTS")
print("========================")

print("Best Threshold:", best_t)

print(
    "Accuracy:",
    accuracy_score(
        y_test_seq,
        final_preds
    )
)

print("\nConfusion Matrix")

print(
    confusion_matrix(
        y_test_seq,
        final_preds
    )
)

print("\nClassification Report")

print(
    classification_report(
        y_test_seq,
        final_preds
    )
)

In [ ]:
# =====================================================
# TCN (TEMPORAL CONVOLUTIONAL NETWORK)
# =====================================================

class TemporalBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        dilation,
        dropout=0.3
    ):

        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.bn1 = nn.BatchNorm1d(out_channels)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.bn2 = nn.BatchNorm1d(out_channels)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        self.downsample = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else None
        )

    def forward(self, x):

        out = self.conv1(x)

        out = out[:, :, :x.size(2)]

        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)

        out = self.conv2(out)

        out = out[:, :, :x.size(2)]

        out = self.bn2(out)
        out = self.relu(out)
        out = self.dropout(out)

        res = x

        if self.downsample is not None:
            res = self.downsample(x)

        return self.relu(out + res)

# =====================================================
# TCN MODEL
# =====================================================

class TCNModel(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.tcn = nn.Sequential(

            TemporalBlock(
                input_size,
                64,
                kernel_size=3,
                dilation=1
            ),

            TemporalBlock(
                64,
                128,
                kernel_size=3,
                dilation=2
            ),

            TemporalBlock(
                128,
                128,
                kernel_size=3,
                dilation=4
            )
        )

        self.fc = nn.Sequential(

            nn.Linear(128, 64),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(64, 1)
        )

    def forward(self, x):

        # (batch, seq, features)
        x = x.permute(0, 2, 1)

        x = self.tcn(x)

        x = x[:, :, -1]

        return self.fc(x)

# =====================================================
# MODEL
# =====================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = TCNModel(
    X_train_seq.shape[2]
).to(device)

# =====================================================
# LOSS
# =====================================================

criterion = nn.BCEWithLogitsLoss()

# =====================================================
# OPTIMIZER
# =====================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

# =====================================================
# TRAINING
# =====================================================

EPOCHS = 30

for epoch in range(EPOCHS):

    model.train()

    loss_sum = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        out = model(xb).squeeze()

        loss = criterion(
            out,
            yb
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        loss_sum += loss.item()

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Loss: {loss_sum/len(train_loader):.4f}"
    )

# =====================================================
# PREDICTION
# =====================================================

model.eval()

preds = []

with torch.no_grad():

    for xb, _ in test_loader:

        xb = xb.to(device)

        p = torch.sigmoid(
            model(xb)
        ).cpu().numpy()

        preds.extend(p)

preds = np.array(preds).flatten()

# =====================================================
# THRESHOLD SEARCH
# =====================================================

best_t = 0.5
best_acc = 0

for t in np.arange(0.30, 0.71, 0.01):

    p = (preds > t).astype(int)

    acc = accuracy_score(
        y_test_seq,
        p
    )

    if acc > best_acc:

        best_acc = acc
        best_t = t

final_preds = (
    preds > best_t
).astype(int)

# =====================================================
# RESULTS
# =====================================================

print("\n========================")
print("TCN RESULTS")
print("========================")

print("Best Threshold:", best_t)

print(
    "Accuracy:",
    accuracy_score(
        y_test_seq,
        final_preds
    )
)

print("\nConfusion Matrix")

print(
    confusion_matrix(
        y_test_seq,
        final_preds
    )
)

print("\nClassification Report")

print(
    classification_report(
        y_test_seq,
        final_preds
    )
)

In [ ]:
# =====================================================
# CNN + TST (OPTIMIZED HIGH ACCURACY VERSION)
# =====================================================

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :].to(x.device)


class CNN_TST(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        # =================================================
        # CNN FEATURE EXTRACTOR (IMPROVED)
        # =================================================

        self.cnn1 = nn.Conv1d(input_size, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)

        self.cnn2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)

        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.2)

        # =================================================
        # PROJECTION
        # =================================================

        self.proj = nn.Linear(128, 128)

        # =================================================
        # POSITIONAL ENCODING (IMPORTANT FIX)
        # =================================================

        self.pos = PositionalEncoding(128)

        # =================================================
        # TRANSFORMER (STABILIZED)
        # =================================================

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            dim_feedforward=256,
            dropout=0.2,
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # =================================================
        # ATTENTION POOLING (STABLE)
        # =================================================

        self.attn = nn.Sequential(
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        # =================================================
        # CLASSIFIER
        # =================================================

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):

        # x: (B, seq, features)

        x = x.permute(0, 2, 1)

        # CNN BLOCK
        x = self.relu(self.bn1(self.cnn1(x)))
        x = self.drop(x)

        x = self.relu(self.bn2(self.cnn2(x)))
        x = self.drop(x)

        x = x.permute(0, 2, 1)

        # PROJECTION
        x = self.proj(x)

        # POSITIONAL INFO
        x = self.pos(x)

        # TRANSFORMER
        x = self.transformer(x)

        # ATTENTION POOLING
        weights = torch.softmax(self.attn(x), dim=1)

        context = torch.sum(weights * x, dim=1)

        return self.fc(context)


# =====================================================
# MODEL SETUP
# =====================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_TST(X_train_seq.shape[2]).to(device)


# =====================================================
# LOSS
# =====================================================

criterion = nn.BCEWithLogitsLoss()


# =====================================================
# OPTIMIZER (IMPORTANT TUNING)
# =====================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.00025,
    weight_decay=1e-5
)


# =====================================================
# TRAINING
# =====================================================

EPOCHS = 35

for epoch in range(EPOCHS):

    model.train()

    loss_sum = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        out = model(xb).squeeze()

        loss = criterion(out, yb)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        loss_sum += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {loss_sum/len(train_loader):.4f}")


# =====================================================
# PREDICTION
# =====================================================

model.eval()

preds = []

with torch.no_grad():

    for xb, _ in test_loader:

        xb = xb.to(device)

        p = torch.sigmoid(model(xb)).cpu().numpy()

        preds.extend(p)

preds = np.array(preds).flatten()


# =====================================================
# THRESHOLD SEARCH
# =====================================================

best_t = 0.5
best_acc = 0

for t in np.arange(0.30, 0.71, 0.01):

    p = (preds > t).astype(int)

    acc = accuracy_score(y_test_seq, p)

    if acc > best_acc:

        best_acc = acc
        best_t = t

final_preds = (preds > best_t).astype(int)


# =====================================================
# RESULTS
# =====================================================

print("\n========================")
print("IMPROVED CNN + TST RESULTS")
print("========================")

print("Best Threshold:", best_t)

print("Accuracy:", accuracy_score(y_test_seq, final_preds))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_seq, final_preds))

print("\nClassification Report")
print(classification_report(y_test_seq, final_preds))

In [38]:
import torch
import torch.nn as nn

# =====================================================
# N-BEATS FOR BINARY CLASSIFICATION
# =====================================================

class NBeatsBlock(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=256,
        n_layers=4
    ):

        super().__init__()

        layers = []

        layers.append(
            nn.Linear(input_dim, hidden_dim)
        )

        layers.append(nn.ReLU())

        for _ in range(n_layers - 1):

            layers.append(
                nn.Linear(hidden_dim, hidden_dim)
            )

            layers.append(
                nn.ReLU()
            )

        self.fc = nn.Sequential(*layers)

        self.backcast = nn.Linear(
            hidden_dim,
            input_dim
        )

        self.forecast = nn.Linear(
            hidden_dim,
            hidden_dim
        )

    def forward(self, x):

        h = self.fc(x)

        backcast = self.backcast(h)

        forecast = self.forecast(h)

        return backcast, forecast


# =====================================================
# N-BEATS MODEL
# =====================================================

class NBeatsClassifier(nn.Module):

    def __init__(
        self,
        seq_len,
        num_features,
        hidden_dim=256,
        n_blocks=3
    ):

        super().__init__()

        self.input_dim = (
            seq_len * num_features
        )

        self.blocks = nn.ModuleList([

            NBeatsBlock(
                self.input_dim,
                hidden_dim
            )

            for _ in range(n_blocks)

        ])

        self.classifier = nn.Sequential(

            nn.Linear(
                hidden_dim,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                64,
                1
            )
        )

    def forward(self, x):

        batch_size = x.size(0)

        x = x.reshape(
            batch_size,
            -1
        )

        residual = x

        forecast_sum = 0

        for block in self.blocks:

            backcast, forecast = block(
                residual
            )

            residual = (
                residual - backcast
            )

            forecast_sum = (
                forecast_sum + forecast
            )

        return self.classifier(
            forecast_sum
        )


# =====================================================
# MODEL
# =====================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = NBeatsClassifier(
    seq_len=X_train_seq.shape[1],
    num_features=X_train_seq.shape[2],
    hidden_dim=256,
    n_blocks=3
).to(device)


# =====================================================
# LOSS
# =====================================================

criterion = nn.BCEWithLogitsLoss()


# =====================================================
# OPTIMIZER
# =====================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-5
)


# =====================================================
# TRAINING
# =====================================================

EPOCHS = 30

for epoch in range(EPOCHS):

    model.train()

    loss_sum = 0

    for xb, yb in train_loader:

        xb = xb.to(device)

        yb = yb.to(device)

        optimizer.zero_grad()

        out = model(xb).squeeze()

        loss = criterion(
            out,
            yb
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        loss_sum += loss.item()

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Loss: {loss_sum/len(train_loader):.4f}"
    )


# =====================================================
# PREDICTION
# =====================================================

model.eval()

preds = []

with torch.no_grad():

    for xb, _ in test_loader:

        xb = xb.to(device)

        p = torch.sigmoid(
            model(xb)
        ).cpu().numpy()

        preds.extend(p)

preds = np.array(
    preds
).flatten()


# =====================================================
# THRESHOLD SEARCH
# =====================================================

best_t = 0.5
best_acc = 0

for t in np.arange(
    0.30,
    0.71,
    0.01
):

    p = (
        preds > t
    ).astype(int)

    acc = accuracy_score(
        y_test_seq,
        p
    )

    if acc > best_acc:

        best_acc = acc
        best_t = t

final_preds = (
    preds > best_t
).astype(int)


# =====================================================
# RESULTS
# =====================================================

print("\n========================")
print("N-BEATS RESULTS")
print("========================")

print(
    "Best Threshold:",
    best_t
)

print(
    "Accuracy:",
    accuracy_score(
        y_test_seq,
        final_preds
    )
)

print(
    "\nConfusion Matrix"
)

print(
    confusion_matrix(
        y_test_seq,
        final_preds
    )
)

print(
    "\nClassification Report"
)

print(
    classification_report(
        y_test_seq,
        final_preds
    )
)

Epoch 1/30 Loss: 0.5954
Epoch 2/30 Loss: 0.4799
Epoch 3/30 Loss: 0.4439
Epoch 4/30 Loss: 0.4000
Epoch 5/30 Loss: 0.3423
Epoch 6/30 Loss: 0.2774
Epoch 7/30 Loss: 0.2034
Epoch 8/30 Loss: 0.1516
Epoch 9/30 Loss: 0.1030
Epoch 10/30 Loss: 0.0791
Epoch 11/30 Loss: 0.0767
Epoch 12/30 Loss: 0.0573
Epoch 13/30 Loss: 0.0437
Epoch 14/30 Loss: 0.0408
Epoch 15/30 Loss: 0.0371
Epoch 16/30 Loss: 0.0342
Epoch 17/30 Loss: 0.0336
Epoch 18/30 Loss: 0.0265
Epoch 19/30 Loss: 0.0285
Epoch 20/30 Loss: 0.0209
Epoch 21/30 Loss: 0.0292
Epoch 22/30 Loss: 0.0199
Epoch 23/30 Loss: 0.0229
Epoch 24/30 Loss: 0.0206
Epoch 25/30 Loss: 0.0176
Epoch 26/30 Loss: 0.0180
Epoch 27/30 Loss: 0.0217
Epoch 28/30 Loss: 0.0145
Epoch 29/30 Loss: 0.0158
Epoch 30/30 Loss: 0.0128

N-BEATS RESULTS
Best Threshold: 0.3
Accuracy: 0.702054171968733

Confusion Matrix
[[2171  563]
 [1076 1691]]

Classification Report
              precision    recall  f1-score   support

           0       0.67      0.79      0.73      2734
           1     